In [ ]:
import cv2
import torch

In [ ]:
from moge.model.v2 import MoGeModel # Let's try MoGe-2

device = torch.device("cuda")

In [ ]:
# Load the model from huggingface hub (or load from local).
model = MoGeModel.from_pretrained("Ruicheng/moge-2-vitl-normal").to(device)                             

# Read the input image and convert to tensor (3, H, W) with RGB values normalized to [0, 1]
input_image = cv2.cvtColor(cv2.imread("1.jpeg"), cv2.COLOR_BGR2RGB)                       
input_image = torch.tensor(input_image / 255, dtype=torch.float32, device=device).permute(2, 0, 1)    

# Infer 
output = model.infer(input_image)
"""
`output` has keys "points", "depth", "mask", "normal" (optional) and "intrinsics",
The maps are in the same size as the input image. 
{
    "points": (H, W, 3),    # point map in OpenCV camera coordinate system (x right, y down, z forward). For MoGe-2, the point map is in metric scale.
    "depth": (H, W),        # depth map
    "normal": (H, W, 3)     # normal map in OpenCV camera coordinate system. (available for MoGe-2-normal)
    "mask": (H, W),         # a binary mask for valid pixels. 
    "intrinsics": (3, 3),   # normalized camera intrinsics
}
"""

In [4]:
import torch
import cv2
import numpy as np
import open3d as o3d
from moge.model.v2 import MoGeModel # Let's try MoGe-2

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Load model
# -----------------------------
model = MoGeModel.from_pretrained("Ruicheng/moge-2-vitl-normal").to(device)                             

# -----------------------------
# Read and preprocess image (lightweight)
# -----------------------------
input_image_bgr = cv2.imread("1.jpeg")

# Resize for faster inference
scale_factor = 0.25  # 1/4 resolution
h, w = input_image_bgr.shape[:2]
input_image_small = cv2.resize(input_image_bgr, (int(w*scale_factor), int(h*scale_factor)))
input_image_rgb = cv2.cvtColor(input_image_small, cv2.COLOR_BGR2RGB)
input_tensor = torch.tensor(input_image_rgb / 255, dtype=torch.float32, device=device).permute(2, 0, 1)

# -----------------------------
# Inference
# -----------------------------
output = model.infer(input_tensor)

points = output["points"]      # (H, W, 3)
normals = output["normal"]     # (H, W, 3)
mask = output["mask"]

# -----------------------------
# Extract valid points and downsample early
# -----------------------------
mask_cpu = mask.cpu().numpy().flatten()
valid_idx = np.where(mask_cpu)[0]

# Sample a smaller subset for visualization
num_samples = min(3000, len(valid_idx))
sample_idx = np.random.choice(valid_idx, size=num_samples, replace=False)

# Flatten arrays and select samples
points_valid = points.cpu().numpy().reshape(-1,3)[sample_idx]
normals_valid = normals.cpu().numpy().reshape(-1,3)[sample_idx]
colors_valid = input_image_rgb.reshape(-1,3)[sample_idx] / 255.0

# Flip Y axis for OpenCV → Cartesian conversion
points_valid[:,1] = -points_valid[:,1]
normals_valid[:,1] = -normals_valid[:,1]

# -----------------------------
# Create Open3D point cloud
# -----------------------------
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_valid)
pcd.colors = o3d.utility.Vector3dVector(colors_valid)

# -----------------------------
# Optional: draw normals as line segments with tip color
# -----------------------------
draw_normals = True
if draw_normals:
    lines = []
    line_colors = []
    arrow_length = 0.05 * np.linalg.norm(points_valid.max(axis=0) - points_valid.min(axis=0))
    points_for_lines = []

    for i in range(len(points_valid)):
        start = points_valid[i]
        end = start + normals_valid[i] * arrow_length
        points_for_lines.append(start)
        points_for_lines.append(end)
        lines.append([2*i, 2*i+1])
        line_colors.append(colors_valid[i])  # tip color

    line_set = o3d.geometry.LineSet(
        points=o3d.utility.Vector3dVector(points_for_lines),
        lines=o3d.utility.Vector2iVector(lines)
    )
    line_set.colors = o3d.utility.Vector3dVector(line_colors)

    # Visualize point cloud + normals
    o3d.visualization.draw_geometries([pcd, line_set])
else:
    # Visualize only point cloud
    o3d.visualization.draw_geometries([pcd])
